In [7]:
%pip install -q langchain langchain-google-genai

In [8]:
import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.3
)

parser = StrOutputParser()

print("LLM is ready!")

Enter your Gemini API key: ··········
LLM is ready!


In [9]:
analysis_prompt = ChatPromptTemplate.from_template("""
You are an email request analyst.

Analyze the following user request and identify:
1. Email purpose
2. Recipient
3. Important details
4. Requested outcome

Do not write the email yet.

User request:
{email_request}
""")

analysis_chain = analysis_prompt | llm | parser

In [10]:
email_request = """
Write an email to my professor asking for an extension
on my assignment because I have two exams this week.
Use a formal tone.
"""

analysis = analysis_chain.invoke({
    "email_request": email_request
})

print(analysis)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Here is the analysis of the user request:

1. **Email purpose:** To request an extension on an upcoming assignment.
2. **Recipient:** A professor.
3. **Important details:** 
   * The user has two exams this week.
   * Tone must be formal.
   * (Implicit: Specific assignment name/due date and requested new deadline will need to be included or placeholder-ed).
4. **Requested outcome:** Approval for a deadline extension on the assignment.


In [11]:
draft_prompt = ChatPromptTemplate.from_template("""
You are a professional email writer.

Write an email draft based on the information below.

Original request:
{email_request}

Request analysis:
{analysis}

Write the email draft only.
Do not add a subject line yet.
""")

drafting_chain = draft_prompt | llm | parser

In [12]:
tone_prompt = ChatPromptTemplate.from_template("""
You are an email tone editor.

Rewrite the email using the requested tone:
{tone}

Preserve the original meaning and important details.
Do not invent facts.

Email draft:
{draft}

Return only the revised email body.
""")

tone_chain = tone_prompt | llm | parser

In [13]:
format_prompt = ChatPromptTemplate.from_template("""
You are an email formatting assistant.

Format the following email into this exact structure:

Subject: [concise subject]

Body:
[complete email body, including an appropriate greeting
and closing]

Revised email:
{adjusted_email}

Return only the formatted email.
""")

formatting_chain = format_prompt | llm | parser

In [14]:
workflow = (
    RunnablePassthrough.assign(
        analysis=lambda x: analysis_chain.invoke({
            "email_request": x["email_request"]
        })
    )
    .assign(
        draft=lambda x: drafting_chain.invoke({
            "email_request": x["email_request"],
            "analysis": x["analysis"]
        })
    )
    .assign(
        adjusted_email=lambda x: tone_chain.invoke({
            "draft": x["draft"],
            "tone": x["tone"]
        })
    )
    .assign(
        final_email=lambda x: formatting_chain.invoke({
            "adjusted_email": x["adjusted_email"]
        })
    )
)

print("Sequential workflow created successfully!")

Sequential workflow created successfully!


In [15]:
result = workflow.invoke({
    "email_request": email_request,
    "tone": "formal"
})

print("=== INPUT ANALYSIS ===")
print(result["analysis"])

print("\n=== EMAIL DRAFT ===")
print(result["draft"])

print("\n=== TONE-ADJUSTED EMAIL ===")
print(result["adjusted_email"])

print("\n=== FINAL EMAIL ===")
print(result["final_email"])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_

=== INPUT ANALYSIS ===
Based on the user request, here is the analysis:

1. **Email purpose:** To request an extension on an assignment due to a heavy exam workload.
2. **Recipient:** The user's professor.
3. **Important details:** 
   * The user has two exams scheduled for this week.
   * The tone must be formal.
4. **Requested outcome:** Granting an extension on the upcoming assignment.

=== EMAIL DRAFT ===
Dear Professor [Last Name],

I hope this email finds you well. 

I am writing to respectfully request a short extension on the upcoming [Assignment Name], which is currently due on [Current Due Date]. 

This week is particularly challenging for me academically, as I have two major exams scheduled. Due to the intensive preparation required for these assessments, I am concerned I will not be able to dedicate the necessary time to complete this assignment to the best of my ability. 

Would it be possible to submit my assignment by [Proposed New Date]? I want to ensure I submit work t